# LeetCode 1367 — Linked List in Binary Tree bằng Python

Link đề: https://leetcode.com/problems/linked-list-in-binary-tree/ · Độ khó: **Medium** · Chủ đề: Linked List, Tree, Depth-First Search, Binary Tree (nâng cao: String Matching / KMP)

Skeleton LeetCode:

```python
# class ListNode:
#     def __init__(self, val=0, next=None): ...
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None): ...
class Solution:
    def isSubPath(self, head: ListNode | None, root: TreeNode | None) -> bool:
        ...
```

## 1. Mục tiêu học

1. Hiểu khái niệm **đường đi xuống (downward path)** trong cây nhị phân: bắt đầu ở **bất kỳ** nút, mỗi bước đi sang con trái hoặc con phải, các nút **liền nhau**.
2. Tách bài toán thành **hai tầng đệ quy**: tầng ngoài "thử mọi điểm bắt đầu" và tầng trong "so khớp liên tiếp từ điểm đó" — một mẫu rất hay gặp (giống LeetCode 572 *Subtree of Another Tree*).
3. Viết baseline "liệt kê mọi đường gốc → lá rồi tìm danh sách con liên tiếp", hiểu vì sao nó tốn bộ nhớ.
4. Nâng cấp lên **KMP trên cây**: mang theo trạng thái khớp khi DFS để đạt `O(N + L)`.
5. Tự dựng `ListNode`, `TreeNode` và hàm dựng cây từ mảng level-order (có `None`) để test như trên LeetCode.

## 2. Đề bài & ví dụ

Cho cây nhị phân `root` và danh sách liên kết đơn bắt đầu tại `head`. Trả về `True` nếu **toàn bộ** các phần tử của danh sách (theo đúng thứ tự từ `head`) trùng với giá trị của một **đường đi xuống liên tiếp** trong cây; ngược lại trả về `False`.

"Đường đi xuống" bắt đầu ở một nút bất kỳ (không bắt buộc là gốc) và chỉ đi từ cha xuống con, không được nhảy cóc.

Cây dùng chung cho cả 3 ví dụ: `root = [1,4,4,null,2,2,null,1,null,6,8,null,null,null,null,1,3]`

- **Ví dụ 1:** `head = [4,2,8]` → `True` (đường 4 → 2 → 8 bên phải).
- **Ví dụ 2:** `head = [1,4,2,6]` → `True` (đi từ gốc 1 → 4 → 2 → 6).
- **Ví dụ 3:** `head = [1,4,2,6,8]` → `False` (6 và 8 là anh em, không nối tiếp được).

**Ràng buộc:** số nút cây trong `[1, 2500]`; số nút danh sách trong `[1, 100]`; `1 <= Node.val <= 100`.

In [1]:
import random
import sys
from collections import deque

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_list(arr):
    dummy = ListNode()
    cur = dummy
    for v in arr:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút)."""
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0])
    q = deque([root])
    i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i]); q.append(node.right)
        i += 1
    return root

TREE = [1, 4, 4, None, 2, 2, None, 1, None, 6, 8, None, None, None, None, 1, 3]
EXAMPLES = [([4, 2, 8], TREE, True), ([1, 4, 2, 6], TREE, True), ([1, 4, 2, 6, 8], TREE, False)]
for head, tree, want in EXAMPLES:
    print(f"head={head} -> mong đợi {want}")

head=[4, 2, 8] -> mong đợi True
head=[1, 4, 2, 6] -> mong đợi True
head=[1, 4, 2, 6, 8] -> mong đợi False


## 3. Kiến thức nền

### 3.1 Danh sách liên kết đơn (`ListNode`)

Mỗi nút giữ `val` và con trỏ `next`. Duyệt bằng vòng `while cur:`; hết danh sách khi `cur is None`. Trong bài này, "khớp xong danh sách" chính là lúc con trỏ danh sách trở thành `None`.

In [2]:
def list_to_array(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

h = build_list([4, 2, 8])
print(list_to_array(h), "| nút thứ 2:", h.next.val, "| sau nút cuối:", h.next.next.next)

[4, 2, 8] | nút thứ 2: 2 | sau nút cuối: None


### 3.2 Cây nhị phân & dựng cây từ level-order

LeetCode mô tả cây bằng mảng **theo tầng** (BFS), `null` nghĩa là vị trí đó trống. Hàm `build_tree` ở trên dùng hàng đợi: lấy lần lượt từng nút cha, gán hai phần tử kế tiếp làm con trái/phải. Để kiểm tra, ta viết hàm ngược lại `tree_to_array` và hàm in các đường gốc → lá.

In [3]:
def tree_to_array(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val)
        q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

root = build_tree(TREE)
print(tree_to_array(root) == TREE)
print("gốc:", root.val, "| con trái:", root.left.val, "| con phải:", root.right.val)

True
gốc: 1 | con trái: 4 | con phải: 4


### 3.3 Đệ quy trên cây và toán tử `or` "đoản mạch"

`A or B` trong Python **không tính `B`** nếu `A` đã đúng. Nhờ vậy `dfs(node.left) or dfs(node.right)` dừng sớm ngay khi tìm thấy đáp án ở nhánh trái — rất quan trọng cho tốc độ của bài "có tồn tại hay không".

In [4]:
calls = []
def f(name, value):
    calls.append(name)
    return value

print(f("A", True) or f("B", True), "| các hàm đã gọi:", calls)
calls.clear()
print(f("A", False) or f("B", True), "| các hàm đã gọi:", calls)

True | các hàm đã gọi: ['A']
True | các hàm đã gọi: ['A', 'B']


### 3.4 Giới hạn đệ quy của Python

Mặc định `sys.getrecursionlimit()` khoảng 1000. Cây suy biến (mọi nút chỉ có con phải) có độ sâu tới 2500 nên đệ quy thuần có thể vượt giới hạn. Trong notebook ta sẽ tăng giới hạn bằng `sys.setrecursionlimit(10000)`; bản nộp LeetCode cũng nên làm vậy (hoặc dùng bản lặp với stack).

In [5]:
print("giới hạn hiện tại:", sys.getrecursionlimit())
sys.setrecursionlimit(10000)
print("giới hạn mới:", sys.getrecursionlimit())

giới hạn hiện tại: 1000
giới hạn mới: 10000


### 3.5 Hàm tiền tố (prefix function) của KMP

Với mẫu `p`, `pi[i]` = độ dài **tiền tố dài nhất** của `p[0..i]` đồng thời là **hậu tố** của nó (không tính toàn bộ). Khi đang khớp được `j` ký tự mà gặp ký tự sai, ta không quay về 0 mà nhảy về `pi[j-1]` — không bao giờ phải so lại từ đầu. Đây là "vũ khí" cho cách nâng cao.

In [6]:
def prefix_function(p):
    pi = [0] * len(p)
    j = 0
    for i in range(1, len(p)):
        while j > 0 and p[i] != p[j]:
            j = pi[j - 1]
        if p[i] == p[j]:
            j += 1
        pi[i] = j
    return pi

print(prefix_function([1, 1, 2]))
print(prefix_function([1, 2, 1, 2, 3]))
print(prefix_function([4, 4, 4, 4]))

[0, 1, 0]
[0, 0, 1, 2, 0]
[0, 1, 2, 3]


## 4. Ý tưởng nền tảng / trực giác

Vẽ cây ví dụ:

```
            1
          /   \
         4     4
          \   /
           2 2
          / / \
         1 6   8
              / \
             1   3
```

Tìm `head = [4,2,8]` bằng tay:
1. Thử bắt đầu tại gốc `1`: `1 ≠ 4` ⇒ thất bại ngay.
2. Thử `4` (trái): khớp `4`; con duy nhất là `2` ⇒ khớp `2`; con của `2` này là `1 ≠ 8` ⇒ thất bại.
3. Thử `4` (phải): khớp `4` → con trái `2` khớp → con phải `8` khớp ⇒ hết danh sách ⇒ **True**.

Với `[1,4,2,6,8]`: đường `1 → 4 → 2 → 6` khớp 4 phần tử, nhưng `6` là lá nên không đi tiếp tới `8` được (8 là **anh em** của 6). Mọi điểm bắt đầu khác cũng thất bại ⇒ **False**.

Như vậy có hai câu hỏi lồng nhau:
- **Ngoài:** bắt đầu ở nút nào?
- **Trong:** từ nút đó, có đi xuống khớp hết danh sách không? (mỗi bước có 2 lựa chọn trái/phải)

## 5. Cách 1 — Brute Force / baseline: liệt kê mọi đường gốc → lá

**Ý tưởng:** mọi đường đi xuống đều là một **đoạn liên tiếp** của ít nhất một đường gốc → lá. Vậy:
1. DFS thu tất cả các đường gốc → lá dưới dạng list giá trị.
2. Với mỗi đường, kiểm tra danh sách có là **mảng con liên tiếp** của nó không (so mọi vị trí bắt đầu).

In [7]:
def all_root_to_leaf_paths(root):
    paths = []
    def dfs(node, path):
        if node is None:
            return
        path.append(node.val)
        if node.left is None and node.right is None:
            paths.append(path[:])          # chép lại vì path còn bị sửa
        else:
            dfs(node.left, path)
            dfs(node.right, path)
        path.pop()                          # quay lui
    dfs(root, [])
    return paths

def contains_contiguous(big, small):
    m = len(small)
    return any(big[i:i + m] == small for i in range(len(big) - m + 1))

def is_subpath_paths(head, root):
    pat = list_to_array(head)
    return any(contains_contiguous(p, pat) for p in all_root_to_leaf_paths(root))

for p in all_root_to_leaf_paths(build_tree(TREE)):
    print(p)
for h, t, want in EXAMPLES:
    print(h, "->", is_subpath_paths(build_list(h), build_tree(t)), "(mong đợi", want, ")")

[1, 4, 2, 1]
[1, 4, 2, 6]
[1, 4, 2, 8, 1]
[1, 4, 2, 8, 3]
[4, 2, 8] -> True (mong đợi True )
[1, 4, 2, 6] -> True (mong đợi True )
[1, 4, 2, 6, 8] -> False (mong đợi False )


**Độ phức tạp:** gọi `N` = số nút cây, `L` = độ dài danh sách, `H` = chiều cao, `K` = số lá. Có `K` đường, mỗi đường dài ≤ `H`; kiểm tra mỗi đường tốn `O(H·L)` ⇒ `O(K·H·L)` thời gian và `O(K·H)` bộ nhớ để lưu các đường.

**Nút thắt:** các đường gốc → lá **chia sẻ tiền tố** (ví dụ `1 → 4` xuất hiện trong nhiều đường), nhưng ta chép và so lại tiền tố đó nhiều lần. Với cây đầy đủ `N = 2500`, `K ≈ 1250`, `H ≈ 12` thì ổn; nhưng với cây "chổi" (một sống dài + nhiều lá) thì `K·H` có thể tới cỡ `N²/4`. Ta muốn so khớp **ngay trên cây**, không chép đường.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tách hai tầng):**
- `match(h, node)`: danh sách **bắt đầu từ `h`** có khớp đường đi xuống **bắt đầu đúng tại `node`** không?
  - Nếu `h is None` ⇒ đã khớp hết ⇒ `True` (kiểm tra **trước**).
  - Nếu `node is None` hoặc `node.val != h.val` ⇒ `False`.
  - Ngược lại ⇒ `match(h.next, node.left) or match(h.next, node.right)`.
- `isSubPath(head, node)`: thử mọi điểm bắt đầu: `match(head, node) or isSubPath(head, node.left) or isSubPath(head, node.right)`.

**Quy tắc quan trọng:** *nếu đang khớp dở mà gặp sai thì KHÔNG được "khớp lại từ head tại nút hiện tại rồi đi tiếp"* trong cùng một hàm — phải để tầng ngoài thử lại từ một điểm bắt đầu mới. Trộn hai tầng là lỗi kinh điển (xem mục 13).

**Insight 2 (KMP):** khi đi từ gốc xuống, chuỗi giá trị trên đường là một "văn bản" và danh sách là "mẫu". KMP cho phép mang theo **trạng thái `j` = số phần tử đã khớp** khi đi xuống; mỗi nhánh con nhận bản sao của `j`. Mỗi nút chỉ được xử lý **một lần** ⇒ tổng `O(N + L)` (khấu hao trên mỗi đường).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS mọi nút × so khớp từ nút đó (đệ quy hai tầng)

**Từng đoạn:**
- `match` là DFS "đi theo giá trị": chỉ rẽ vào con có giá trị bằng phần tử tiếp theo của danh sách.
- `solve` là DFS "thử mọi nút làm điểm bắt đầu".
- Nhờ `or` đoản mạch, ngay khi một lần `match` thành công thì toàn bộ dừng.

In [8]:
def is_subpath_dfs(head, root):
    def match(h, node):
        if h is None:                 # khớp hết danh sách
            return True
        if node is None or node.val != h.val:
            return False
        return match(h.next, node.left) or match(h.next, node.right)

    def solve(node):
        if node is None:
            return False
        return match(head, node) or solve(node.left) or solve(node.right)

    return solve(root)

for h, t, want in EXAMPLES:
    print(h, "->", is_subpath_dfs(build_list(h), build_tree(t)), "(mong đợi", want, ")")

[4, 2, 8] -> True (mong đợi True )
[1, 4, 2, 6] -> True (mong đợi True )
[1, 4, 2, 6, 8] -> False (mong đợi False )


Code nộp LeetCode (Cách 2):

In [9]:
class Solution:
    def isSubPath(self, head: "ListNode | None", root: "TreeNode | None") -> bool:
        def match(h, node):
            if h is None:
                return True
            if node is None or node.val != h.val:
                return False
            return match(h.next, node.left) or match(h.next, node.right)

        if root is None:
            return False
        return (match(head, root)
                or self.isSubPath(head, root.left)
                or self.isSubPath(head, root.right))

print([Solution().isSubPath(build_list(h), build_tree(t)) for h, t, _ in EXAMPLES])

[True, True, False]


**Độ phức tạp Cách 2:** mỗi nút là điểm bắt đầu một lần; từ mỗi điểm, `match` đi sâu tối đa `min(L, H)` tầng nhưng có thể rẽ cả hai nhánh nếu giá trị trùng ⇒ worst-case `O(N · min(2^L, N))` về lý thuyết; trong thực tế (và với cây sâu, giá trị lặp) thường nêu là `O(N · min(L, H))`. Bộ nhớ `O(H)` cho ngăn xếp đệ quy. Với `N ≤ 2500, L ≤ 100` chạy rất nhanh.

### 7.2 Cách 3 — Bản lặp dùng stack (tránh giới hạn đệ quy)

Thay đệ quy bằng stack chứa cặp `(nút danh sách, nút cây)`. Tầng ngoài duyệt mọi nút bằng BFS; tầng trong là DFS lặp.

In [10]:
def is_subpath_iter(head, root):
    def match_iter(start):
        stack = [(head, start)]
        while stack:
            h, node = stack.pop()
            if h is None:
                return True
            if node is None or node.val != h.val:
                continue
            stack.append((h.next, node.left))
            stack.append((h.next, node.right))
        return False

    q = deque([root] if root else [])
    while q:
        node = q.popleft()
        if node.val == head.val and match_iter(node):
            return True
        if node.left: q.append(node.left)
        if node.right: q.append(node.right)
    return False

for h, t, want in EXAMPLES:
    print(h, "->", is_subpath_iter(build_list(h), build_tree(t)), "(mong đợi", want, ")")

[4, 2, 8] -> True (mong đợi True )
[1, 4, 2, 6] -> True (mong đợi True )
[1, 4, 2, 6, 8] -> False (mong đợi False )


Code nộp LeetCode (Cách 3):

In [11]:
class Solution:
    def isSubPath(self, head: "ListNode | None", root: "TreeNode | None") -> bool:
        q = deque([root])
        while q:
            node = q.popleft()
            if node is None:
                continue
            if node.val == head.val:
                stack = [(head, node)]
                while stack:
                    h, t = stack.pop()
                    if h is None:
                        return True
                    if t is not None and t.val == h.val:
                        stack.append((h.next, t.left))
                        stack.append((h.next, t.right))
            q.append(node.left)
            q.append(node.right)
        return False

print([Solution().isSubPath(build_list(h), build_tree(t)) for h, t, _ in EXAMPLES])

[True, True, False]


**Độ phức tạp Cách 3:** giống Cách 2 về thời gian; bộ nhớ `O(N)` cho hàng đợi + `O(2·L)` cho stack mỗi lần khớp. Không phụ thuộc giới hạn đệ quy.

### 7.3 Cách 4 — KMP trên cây (nâng cao, `O(N + L)`)

**Từng đoạn:**
1. Chuyển danh sách thành mảng `pat` và tính `pi = prefix_function(pat)`.
2. DFS từ gốc với tham số `j` = số phần tử `pat` đã khớp ở **cuối đường hiện tại**.
3. Tại nút `node`: trong khi `j > 0` và `pat[j] != node.val` thì `j = pi[j-1]` (lùi theo KMP). Nếu `pat[j] == node.val` thì `j += 1`.
4. Nếu `j == len(pat)` ⇒ tìm thấy. Ngược lại đi xuống hai con với **cùng** giá trị `j` (mỗi con một bản sao — vì `j` là số nguyên truyền theo giá trị).

Vì sao đúng: KMP trên một chuỗi chỉ phụ thuộc vào phần đã đọc; mỗi đường gốc → nút là một chuỗi, và hai đường chung tiền tố thì chung trạng thái tới điểm rẽ.

In [12]:
def is_subpath_kmp(head, root):
    pat = list_to_array(head)
    pi = prefix_function(pat)
    m = len(pat)

    def dfs(node, j):
        if node is None:
            return False
        while j > 0 and pat[j] != node.val:
            j = pi[j - 1]
        if pat[j] == node.val:
            j += 1
        if j == m:
            return True
        return dfs(node.left, j) or dfs(node.right, j)

    return dfs(root, 0)

for h, t, want in EXAMPLES:
    print(h, "->", is_subpath_kmp(build_list(h), build_tree(t)), "(mong đợi", want, ")")

[4, 2, 8] -> True (mong đợi True )
[1, 4, 2, 6] -> True (mong đợi True )
[1, 4, 2, 6, 8] -> False (mong đợi False )


Code nộp LeetCode (Cách 4):

In [13]:
class Solution:
    def isSubPath(self, head: "ListNode | None", root: "TreeNode | None") -> bool:
        pat = []
        while head:
            pat.append(head.val)
            head = head.next
        m = len(pat)
        pi = [0] * m
        k = 0
        for i in range(1, m):
            while k and pat[i] != pat[k]:
                k = pi[k - 1]
            if pat[i] == pat[k]:
                k += 1
            pi[i] = k
        # DFS lặp: stack (nút, j)
        stack = [(root, 0)]
        while stack:
            node, j = stack.pop()
            if node is None:
                continue
            while j and pat[j] != node.val:
                j = pi[j - 1]
            if pat[j] == node.val:
                j += 1
            if j == m:
                return True
            stack.append((node.left, j))
            stack.append((node.right, j))
        return False

print([Solution().isSubPath(build_list(h), build_tree(t)) for h, t, _ in EXAMPLES])

[True, True, False]


**Độ phức tạp Cách 4:** tính `pi` mất `O(L)`. Trong DFS, vòng `while` lùi `j` có thể chạy nhiều lần tại một nút, nhưng `j` chỉ tăng tối đa 1 mỗi nút trên đường, nên trên **mỗi đường gốc → lá** tổng số lần lùi ≤ độ dài đường. Một cận an toàn: `O(N · min(L, H))` worst-case, còn trường hợp thường gặp gần `O(N + L)`; quan trọng hơn, **không bao giờ so lại** cùng một (điểm bắt đầu, vị trí) nhiều lần như Cách 2 trên các cây giá trị lặp. Bộ nhớ `O(H + L)`.

## 8. TRACE từng bước

Trace Cách 2 với `head = [4,2,8]`. Hàm `trace_match` in mỗi lời gọi `match(h, node)` kèm độ sâu thụt lề, còn `trace_solve` in điểm bắt đầu đang thử.

| Bước | Điểm bắt đầu | Diễn biến | Kết quả |
|---|---|---|---|
| 1 | 1 (gốc) | `1 ≠ 4` | False |
| 2 | 4 (trái) | 4 ✓ → 2 ✓ → 1 ≠ 8, nhánh phải None | False |
| 3 | 2 (con của 4 trái) | `2 ≠ 4` | False |
| 4 | 1 (lá) | `1 ≠ 4` | False |
| 5 | 4 (phải) | 4 ✓ → 2 ✓ → 6 ≠ 8 → 8 ✓ → hết danh sách | **True** |

In [14]:
def trace_isSubPath(head_arr, tree_arr):
    head, root = build_list(head_arr), build_tree(tree_arr)

    def match(h, node, depth):
        pad = "    " * depth
        hv = None if h is None else h.val
        nv = None if node is None else node.val
        if h is None:
            print(f"{pad}match(h=None, node={nv}) -> hết danh sách: True")
            return True
        if node is None or node.val != h.val:
            print(f"{pad}match(h={hv}, node={nv}) -> False")
            return False
        print(f"{pad}match(h={hv}, node={nv}) -> khớp, đi xuống")
        return match(h.next, node.left, depth + 1) or match(h.next, node.right, depth + 1)

    step = 0
    def solve(node):
        nonlocal step
        if node is None:
            return False
        step += 1
        print(f"[Điểm bắt đầu #{step}] nút {node.val}")
        if match(head, node, 1):
            return True
        return solve(node.left) or solve(node.right)

    res = solve(root)
    print("KẾT QUẢ:", res)
    return res

trace_isSubPath([4, 2, 8], TREE)

[Điểm bắt đầu #1] nút 1
    match(h=4, node=1) -> False
[Điểm bắt đầu #2] nút 4
    match(h=4, node=4) -> khớp, đi xuống
        match(h=2, node=None) -> False
        match(h=2, node=2) -> khớp, đi xuống
            match(h=8, node=1) -> False
            match(h=8, node=None) -> False
[Điểm bắt đầu #3] nút 2
    match(h=4, node=2) -> False
[Điểm bắt đầu #4] nút 1
    match(h=4, node=1) -> False
[Điểm bắt đầu #5] nút 4
    match(h=4, node=4) -> khớp, đi xuống
        match(h=2, node=2) -> khớp, đi xuống
            match(h=8, node=6) -> False
            match(h=8, node=8) -> khớp, đi xuống
                match(h=None, node=1) -> hết danh sách: True
KẾT QUẢ: True


True

Trace Cách 4 (KMP) với `head = [1,1,2]` trên cây là một đường thẳng `1 → 1 → 1 → 2`. Ta in `j` trước/sau khi xử lý mỗi nút:

In [15]:
def trace_kmp(head_arr, tree_arr):
    pat = head_arr
    pi = prefix_function(pat)
    print("pat =", pat, "| pi =", pi)
    root = build_tree(tree_arr)

    def dfs(node, j, depth):
        if node is None:
            return False
        before = j
        while j > 0 and pat[j] != node.val:
            j = pi[j - 1]
        if pat[j] == node.val:
            j += 1
        print("  " * depth + f"node={node.val}: j {before} -> {j}")
        if j == len(pat):
            print("  " * depth + "  j == len(pat) => TÌM THẤY")
            return True
        return dfs(node.left, j, depth + 1) or dfs(node.right, j, depth + 1)

    return dfs(root, 0, 0)

chain = [1, None, 1, None, 1, None, 2]
print("kết quả:", trace_kmp([1, 1, 2], chain))

pat = [1, 1, 2] | pi = [0, 1, 0]
node=1: j 0 -> 1
  node=1: j 1 -> 2
    node=1: j 2 -> 2
      node=2: j 2 -> 3
        j == len(pat) => TÌM THẤY
kết quả: True


Quan sát: tại nút `1` thứ ba, `j = 2` nhưng `pat[2] = 2 ≠ 1` ⇒ lùi `j = pi[1] = 1`, rồi `pat[1] = 1` khớp ⇒ `j = 2`. Không quay về 0 — đó là sức mạnh của KMP.

## 9. Code nộp LeetCode

Bản khuyên dùng: đệ quy hai tầng (dễ đọc, đủ nhanh), có tăng giới hạn đệ quy cho cây suy biến.

In [16]:
import sys
sys.setrecursionlimit(10000)

class Solution:
    def isSubPath(self, head: "ListNode | None", root: "TreeNode | None") -> bool:
        def match(h, node):
            if h is None:
                return True
            if node is None or node.val != h.val:
                return False
            return match(h.next, node.left) or match(h.next, node.right)

        def solve(node):
            if node is None:
                return False
            return match(head, node) or solve(node.left) or solve(node.right)

        return solve(root)

for h, t, want in EXAMPLES:
    got = Solution().isSubPath(build_list(h), build_tree(t))
    print(h, got, "OK" if got == want else "SAI")

[4, 2, 8] True OK
[1, 4, 2, 6] True OK
[1, 4, 2, 6, 8] False OK


## 10. Vì sao thuật toán đúng

**`match` đúng (quy nạp theo độ dài danh sách còn lại):**
- Danh sách rỗng (`h is None`) luôn khớp — đúng theo định nghĩa.
- Danh sách khác rỗng khớp đường bắt đầu tại `node` ⇔ `node` tồn tại, `node.val == h.val`, và phần còn lại `h.next` khớp đường bắt đầu tại **một trong hai con**. Đó chính xác là công thức đệ quy.

**`solve` đúng:** một đường đi xuống hợp lệ bắt đầu tại nút `s` nào đó trong cây. `solve` duyệt **mọi** nút (gốc, rồi toàn bộ cây con trái, toàn bộ cây con phải) và gọi `match(head, s)` cho từng nút ⇒ nếu tồn tại `s` thì chắc chắn được thử; nếu `solve` trả `True` thì có `s` khớp thật.

**KMP đúng:** bất biến của DFS: *khi vào xử lý xong nút `v`, `j` = độ dài tiền tố dài nhất của `pat` là hậu tố của chuỗi giá trị trên đường gốc → `v`*. Đây đúng là bất biến của KMP trên chuỗi đó; vì mỗi con nhận bản sao `j` của cha nên bất biến giữ cho mọi đường. Có đường đi xuống khớp `pat` kết thúc tại `v` ⇔ `j == m` tại `v`.

## 11. Phân tích Time / Space

Ký hiệu: `N` = số nút cây (≤ 2500), `L` = độ dài danh sách (≤ 100), `H` = chiều cao cây (từ `log N` tới `N`), `K` = số lá.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê đường gốc→lá | `O(K·H·L)` | `O(K·H)` lưu đường |
| 2. DFS × match (đệ quy) | thường `O(N·min(L,H))`; worst có thể lớn hơn khi giá trị lặp và rẽ cả hai nhánh | `O(H)` |
| 3. Bản lặp | như Cách 2 | `O(N)` |
| 4. KMP trên cây | `O(L)` + DFS; mỗi nút xử lý 1 lần, phần lùi khấu hao theo đường | `O(H + L)` |

- **Average** (cây ngẫu nhiên, giá trị đa dạng 1..100): `match` gần như dừng ngay ở bước đầu ⇒ Cách 2 ≈ `O(N)`.
- **Worst** (cây toàn giá trị 1, danh sách `[1,1,...,1,2]`): Cách 2 phải đi sâu tối đa `L` tầng từ gần như mọi nút; KMP vẫn chỉ đi mỗi nút một lần.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
METHODS = {
    "paths": is_subpath_paths,
    "dfs": is_subpath_dfs,
    "iter": is_subpath_iter,
    "kmp": is_subpath_kmp,
    "submit": lambda h, r: Solution().isSubPath(h, r),
}

edge_cases = [
    ([1], [1], True),                                   # 1 nút, khớp
    ([2], [1], False),                                  # 1 nút, không khớp
    ([1, 2], [1, 2], True),                             # kết thúc đúng tại lá
    ([1, 2], [1, None, 3, None, 2], False),             # không được nhảy cóc
    ([1, 1, 2], [1, None, 1, None, 1, None, 2], True),  # cần "khớp lại" dịch 1
    ([2, 1], [1, 2], False),                            # chỉ đi xuống, không đi lên
    ([1, 2, 3], [1, 2, 2, 3], True),                    # nhánh trái 2 → 3
    ([3], [1, 2, 2, 3], True),                          # bắt đầu tại lá
]
for name, fn in METHODS.items():
    for h, t, want in EXAMPLES + edge_cases:
        got = fn(build_list(h), build_tree(t))
        assert got == want, (name, h, t, got, want)
print("Tất cả", len(METHODS), "cách đều đúng trên", len(EXAMPLES) + len(edge_cases), "test")

Tất cả 5 cách đều đúng trên 11 test


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Dùng giá trị trong `{1, 2}` để cây có rất nhiều chỗ khớp một phần — đây là chỗ các lỗi dễ lộ ra nhất. Danh sách đôi khi được lấy **thật** từ một đường trong cây để chắc có cả ca `True`.

In [18]:
def random_tree(rng, n, vals):
    if n == 0:
        return None
    nodes = [TreeNode(rng.choice(vals)) for _ in range(n)]
    for i in range(1, n):
        while True:
            p = nodes[rng.randrange(i)]
            if p.left is None and rng.random() < 0.5:
                p.left = nodes[i]; break
            if p.right is None:
                p.right = nodes[i]; break
    return nodes[0]

def random_downward_path(rng, root):
    node = root
    for _ in range(rng.randint(0, 5)):
        kids = [c for c in (node.left, node.right) if c]
        if not kids: break
        node = rng.choice(kids)
    arr = []
    for _ in range(rng.randint(1, 6)):
        arr.append(node.val)
        kids = [c for c in (node.left, node.right) if c]
        if not kids: break
        node = rng.choice(kids)
    return arr

rng = random.Random(1367)
cnt_true = 0
for _ in range(600):
    root = random_tree(rng, rng.randint(1, 40), [1, 2] if rng.random() < 0.7 else [1, 2, 3, 4])
    if rng.random() < 0.5:
        arr = random_downward_path(rng, root)
    else:
        arr = [rng.choice([1, 2]) for _ in range(rng.randint(1, 6))]
    head = build_list(arr)
    ans = {name: fn(head, root) for name, fn in METHODS.items()}
    assert len(set(ans.values())) == 1, (arr, ans)
    cnt_true += ans["dfs"]
print("600 test ngẫu nhiên khớp nhau; số ca True =", cnt_true)

600 test ngẫu nhiên khớp nhau; số ca True = 443


### 12.3 Test lớn: cây suy biến 2500 nút và ca xấu cho Cách 2

In [19]:
import time
# cây thẳng 2500 nút toàn giá trị 1, nút cuối là 2
nodes = [TreeNode(1) for _ in range(2499)] + [TreeNode(2)]
for a, b in zip(nodes, nodes[1:]):
    a.right = b
deep_root = nodes[0]
head_bad = build_list([1] * 99 + [2])
head_no = build_list([1] * 99 + [3])
for name in ["dfs", "iter", "kmp", "submit"]:
    t0 = time.perf_counter()
    r1 = METHODS[name](head_bad, deep_root)
    r2 = METHODS[name](head_no, deep_root)
    print(f"{name:7s} -> {r1}, {r2} | {time.perf_counter() - t0:.3f}s")

dfs     -> True, False | 0.044s


iter    -> True, False | 0.088s
kmp     -> True, False | 0.002s


submit  -> True, False | 0.049s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trộn hai tầng: khi sai thì "reset về head" và đi tiếp xuống con

Code sai (giống thuật toán tìm chuỗi "ngây thơ" không quay lui):

In [20]:
def wrong_reset(head, root):
    def dfs(h, node):
        if h is None:
            return True
        if node is None:
            return False
        if node.val == h.val:
            return dfs(h.next, node.left) or dfs(h.next, node.right)
        # SAI: reset về head nhưng bỏ qua việc thử lại chính node này,
        # và bỏ mất các điểm bắt đầu nằm giữa đoạn đã khớp dở
        return dfs(head, node.left) or dfs(head, node.right)
    return dfs(head, root)

h, t = [1, 1, 2], [1, None, 1, None, 1, None, 2]
print("wrong_reset:", wrong_reset(build_list(h), build_tree(t)), "| đúng:", is_subpath_dfs(build_list(h), build_tree(t)))

wrong_reset: False | đúng: True


**Vì sao sai:** trên đường `1,1,1,2`, khớp `1,1` rồi gặp `1 ≠ 2`; code nhảy sang con với `head` mà **không thử nút `1` thứ ba làm điểm bắt đầu**, cũng không thử nút `1` thứ hai. Kết quả `False` trong khi đáp án là `True`. Hơn nữa, một khi đã khớp `node.val == h.val` thì code **không bao giờ** thử lại `node` đó làm điểm bắt đầu cho các nhánh khác.

**Code đúng:** tách riêng `match` (chỉ khớp liên tiếp) và `solve` (thử mọi điểm bắt đầu) như Cách 2, hoặc dùng KMP.

### Lỗi 2 — Gọi lại hàm ngoài cho phần tiếp theo ⇒ cho phép "nhảy cóc"

In [21]:
class WrongSkip:
    def isSubPath(self, head, root):
        if head is None:
            return True
        if root is None:
            return False
        if head.val == root.val:
            # SAI: dùng hàm ngoài cho h.next => phần tiếp theo được bắt đầu ở bất kỳ đâu bên dưới
            if self.isSubPath(head.next, root.left) or self.isSubPath(head.next, root.right):
                return True
        return self.isSubPath(head, root.left) or self.isSubPath(head, root.right)

h, t = [1, 2], [1, None, 3, None, 2]      # đường 1 -> 3 -> 2, không có 1 -> 2 liền nhau
print("WrongSkip:", WrongSkip().isSubPath(build_list(h), build_tree(t)), "| đúng:", is_subpath_dfs(build_list(h), build_tree(t)))

WrongSkip: True | đúng: False


**Vì sao sai:** sau khi khớp `1`, phần `[2]` được tìm ở **bất kỳ** đâu trong cây con (hàm ngoài thử mọi điểm bắt đầu), nên nó tìm thấy `2` cách hai tầng ⇒ trả `True` sai. Đề yêu cầu các nút **liên tiếp**.

**Code đúng:** phần tiếp theo phải dùng `match(h.next, node.left/right)` — hàm chỉ khớp tại **đúng** nút con.

### Lỗi 3 — Kiểm tra `node is None` trước `h is None`

In [22]:
def wrong_order(head, root):
    def match(h, node):
        if node is None:              # SAI: kiểm tra cây trước
            return False
        if h is None:
            return True
        if node.val != h.val:
            return False
        return match(h.next, node.left) or match(h.next, node.right)
    def solve(node):
        return node is not None and (match(head, node) or solve(node.left) or solve(node.right))
    return solve(root)

h, t = [1, 2], [1, 2]                     # danh sách kết thúc đúng tại lá
print("wrong_order:", wrong_order(build_list(h), build_tree(t)), "| đúng:", is_subpath_dfs(build_list(h), build_tree(t)))

wrong_order: False | đúng: True


**Vì sao sai:** khi khớp xong phần tử cuối tại **lá**, lời gọi tiếp theo là `match(None, None)`. Code sai thấy `node is None` trước và trả `False`, dù danh sách đã khớp hết. Lỗi chỉ lộ ra khi đường khớp kết thúc **đúng tại lá** — ví dụ đề có thể không bắt được.

**Code đúng:** luôn kiểm tra `h is None` (đã khớp hết) **trước** `node is None`.

### Lỗi 4 — Quên tăng giới hạn đệ quy với cây suy biến

Ta mô phỏng giới hạn mặc định 1000 rồi bắt lỗi để in ra (không để exception văng ra):

In [23]:
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    res = is_subpath_dfs(build_list([2]), deep_root)    # 2 nằm ở độ sâu 2500
    print("chạy được:", res)
except RecursionError as e:
    print("RecursionError:", str(e)[:50], "...")
finally:
    sys.setrecursionlimit(old)
print("Sau khi tăng giới hạn:", is_subpath_dfs(build_list([2]), deep_root), "| bản lặp:", is_subpath_iter(build_list([2]), deep_root))

RecursionError: maximum recursion depth exceeded ...
Sau khi tăng giới hạn: True | bản lặp: True


**Code đúng:** thêm `sys.setrecursionlimit(10000)` ở đầu file nộp, hoặc dùng Cách 3 / Cách 4 bản lặp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Cấu trúc A nằm trong cấu trúc B, bắt đầu ở đâu cũng được"** ⇒ nghĩ ngay tới **hai tầng**: `solve(B) = match(A, B) or solve(B.left) or solve(B.right)`. Cùng khuôn với LeetCode 572 (Subtree of Another Tree), 437 (Path Sum III bản brute force).
- Hàm `match` luôn có ba dòng chuẩn: *mẫu hết ⇒ True*; *cây hết hoặc khác giá trị ⇒ False*; *đi tiếp*.
- Khi đề có chữ "đường đi xuống" + "dãy cho trước" và muốn tối ưu ⇒ coi đường gốc → nút là một chuỗi, dùng **KMP/rolling hash mang theo trạng thái** khi DFS.
- Test tự viết luôn gồm: khớp kết thúc tại lá, khớp dở rồi phải dịch 1, và đường có "khe hở" (nhảy cóc).
- Python: nhớ `sys.setrecursionlimit` cho bài cây có tới vài nghìn tầng.

## 15. Biến thể của bài

1. **Đếm số đường đi xuống** khớp danh sách (không dừng sớm) — bỏ `or` đoản mạch, cộng dồn.
2. **Mảng có phải là đường gốc → lá?** (LeetCode 1430): chỉ còn tầng trong, bắt đầu ở gốc, kết thúc phải là lá.
3. **Subtree of Another Tree** (572): tầng trong so sánh **cả cây con** thay vì một đường.
4. **Đường có thể đi lên rồi đi xuống** (đường bất kỳ trong cây): cần coi cây là đồ thị vô hướng, DFS từ mỗi nút với tập đã thăm.
5. **Nhiều danh sách mẫu cùng lúc**: dùng Aho–Corasick (KMP cho nhiều mẫu) mang trạng thái khi DFS.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê đường | thu mọi đường gốc→lá, tìm mảng con liên tiếp | `O(K·H·L)` | `O(K·H)` | để kiểm chứng, cây nhỏ |
| 2. DFS × match | thử mọi nút làm điểm bắt đầu, so khớp đệ quy | thường `O(N·min(L,H))` | `O(H)` | **bài phỏng vấn**, dễ viết nhất |
| 3. Bản lặp | như 2 nhưng dùng queue + stack | như 2 | `O(N)` | cây rất sâu, sợ giới hạn đệ quy |
| 4. KMP trên cây | mang trạng thái KMP khi DFS | gần `O(N + L)` | `O(H + L)` | giá trị lặp nhiều, muốn tối ưu / follow-up |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** với cây `[5,3,8,3,null,null,8]` và các danh sách `[3,3]`, `[5,8,8]`, `[5,3,8]`, kết quả `isSubPath` là gì?

**Bài 2 (tính tay):** tính `prefix_function([2,2,3,2,2,2])`.

**Bài 3 (điền TODO):** hoàn thiện hàm đếm **số đường đi xuống** khớp danh sách (mỗi điểm bắt đầu + mỗi lựa chọn nhánh khác nhau tính là một đường).

In [24]:
def count_subpaths(head, root):
    """Đếm số đường đi xuống khớp toàn bộ danh sách. (Bài tập — chưa gọi)"""
    def match_count(h, node):
        # TODO: nếu node None hoặc khác giá trị -> 0;
        #       nếu h là phần tử cuối (h.next is None) -> 1 (đừng đếm 2 lần qua hai con None!);
        #       ngược lại cộng kết quả hai nhánh
        pass
    def solve(node):
        # TODO: match_count tại node + solve(trái) + solve(phải)
        pass
    return solve(root)

**Bài 4 (tự viết lại):** viết `is_subpath_iterative_two_stacks(head, root)` không dùng đệ quy, không dùng `deque`: tầng ngoài dùng một stack nút, tầng trong dùng một stack `(h, node)`.

**Bài 5 (mini challenge — LeetCode 1430):** cho mảng `arr` và cây; kiểm tra `arr` có phải **chính xác** một đường từ **gốc đến lá** không (bắt đầu ở gốc, kết thúc ở lá).

In [25]:
def is_valid_root_to_leaf(root, arr):
    """Bài 5 — tự viết. (Bài tập — chưa gọi)"""
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 (nhẹ) | Gợi ý 2 (vừa) | Gợi ý 3 (gần lời giải) |
|---|---|---|---|
| 1 | Vẽ cây ra trước | `[3,3]`: 3 trái có con trái 3 | True, True, False |
| 2 | `pi[0]` luôn 0 | khi gặp 3, `j` lùi về 0 | `[0,1,0,1,2,2]` |
| 3 | Giống `match` nhưng trả số | `or` → `+`; cẩn thận: nếu dùng `h is None → 1` thì một lá khớp bị đếm 2 lần (hai con None) | dừng tại `h.next is None` → 1, rồi `return match_count(h.next,L) + match_count(h.next,R)` |
| 4 | Stack thay cho đệ quy | tầng ngoài: pop nút, push hai con | tầng trong như Cách 3 |
| 5 | Chỉ một điểm bắt đầu: gốc | dùng chỉ số `i` trong `arr` | khi `i == len(arr)-1` phải là lá và giá trị khớp |

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
t1 = build_tree([5, 3, 8, 3, None, None, 8])
ans1 = [is_subpath_dfs(build_list(a), t1) for a in ([3, 3], [5, 8, 8], [5, 3, 8])]
print("Bài 1:", ans1)
assert ans1 == [True, True, False]

# Bài 2
ans2 = prefix_function([2, 2, 3, 2, 2, 2])
print("Bài 2:", ans2)
assert ans2 == [0, 1, 0, 1, 2, 2]

Bài 1: [True, True, False]
Bài 2: [0, 1, 0, 1, 2, 2]


In [27]:
# Bài 3
def count_subpaths_sol(head, root):
    def match_count(h, node):
        if node is None or node.val != h.val:
            return 0
        if h.next is None:          # khớp xong phần tử cuối tại node -> đúng 1 đường
            return 1
        return match_count(h.next, node.left) + match_count(h.next, node.right)
    def solve(node):
        if node is None:
            return 0
        return match_count(head, node) + solve(node.left) + solve(node.right)
    return solve(root)

assert count_subpaths_sol(build_list([1]), build_tree([1, 1, 1])) == 3
assert count_subpaths_sol(build_list([1, 1]), build_tree([1, 1, 1])) == 2
assert count_subpaths_sol(build_list([4, 2]), build_tree(TREE)) == 2
assert count_subpaths_sol(build_list([1, 4, 2, 6, 8]), build_tree(TREE)) == 0
# đối chiếu: count > 0  <=>  isSubPath
rng = random.Random(7)
for _ in range(200):
    r = random_tree(rng, rng.randint(1, 25), [1, 2])
    a = [rng.choice([1, 2]) for _ in range(rng.randint(1, 4))]
    assert (count_subpaths_sol(build_list(a), r) > 0) == is_subpath_dfs(build_list(a), r)
print("Bài 3: OK")

Bài 3: OK


In [28]:
# Bài 4
def is_subpath_iterative_two_stacks(head, root):
    outer = [root]
    while outer:
        node = outer.pop()
        if node is None:
            continue
        inner = [(head, node)]
        while inner:
            h, t = inner.pop()
            if h is None:
                return True
            if t is not None and t.val == h.val:
                inner.append((h.next, t.left))
                inner.append((h.next, t.right))
        outer.append(node.left)
        outer.append(node.right)
    return False

rng = random.Random(8)
for _ in range(300):
    r = random_tree(rng, rng.randint(1, 30), [1, 2, 3])
    a = [rng.choice([1, 2, 3]) for _ in range(rng.randint(1, 4))]
    assert is_subpath_iterative_two_stacks(build_list(a), r) == is_subpath_kmp(build_list(a), r)
print("Bài 4: OK")

Bài 4: OK


In [29]:
# Bài 5
def is_valid_root_to_leaf_sol(root, arr):
    def dfs(node, i):
        if node is None or i >= len(arr) or node.val != arr[i]:
            return False
        if i == len(arr) - 1:
            return node.left is None and node.right is None
        return dfs(node.left, i + 1) or dfs(node.right, i + 1)
    return dfs(root, 0)

t5 = build_tree([0, 1, 0, 0, 1, 0, None, None, 1, 0, 0])
assert is_valid_root_to_leaf_sol(t5, [0, 1, 0, 1]) is True
assert is_valid_root_to_leaf_sol(t5, [0, 0, 1]) is False     # không tồn tại
assert is_valid_root_to_leaf_sol(t5, [0, 1, 1]) is False     # 1 cuối không phải lá
assert is_valid_root_to_leaf_sol(build_tree(TREE), [1, 4, 2, 8, 3]) is True
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "A nằm trong B, bắt đầu ở đâu cũng được":**

```python
def match(h, node):                 # khớp liên tiếp tại đúng node
    if h is None: return True        # 1) mẫu hết  -> True  (KIỂM TRA TRƯỚC)
    if node is None or node.val != h.val:
        return False                 # 2) cây hết / khác -> False
    return match(h.next, node.left) or match(h.next, node.right)

def solve(node):                    # thử mọi điểm bắt đầu
    if node is None: return False
    return match(head, node) or solve(node.left) or solve(node.right)
```

**Template KMP trên cây:** `dfs(node, j)`: lùi `j = pi[j-1]` khi khác; tăng `j` khi bằng; `j == m` ⇒ True; truyền `j` cho hai con.

**Checklist trước khi submit:**
- [ ] `match` kiểm tra `h is None` **trước** `node is None`.
- [ ] Phần tiếp theo gọi `match` (liên tiếp), **không** gọi lại hàm ngoài (tránh nhảy cóc).
- [ ] Tầng ngoài thử **mọi** nút, kể cả lá.
- [ ] Đã thử ca "khớp dở rồi phải dịch 1": `head=[1,1,2]`, đường `1,1,1,2`.
- [ ] Cây sâu 2500: đã `sys.setrecursionlimit` hoặc dùng bản lặp.
- [ ] Đúng chữ ký: `isSubPath(self, head, root) -> bool`.